# Experiment 1 — Incremental Learning Benchmark

**PhD Project:** Collaboration Humain-Robot : Apprentissage incrémental et adaptation comportementale  
**Author:** Ameur Gargouri  
**Notebook:** `03_incremental_learning_benchmark.ipynb`  

This self-contained Colab notebook benchmarks **6 continual learning strategies** on the HARMONIC dataset:

| # | Strategy | Family | Paper |
|---|----------|--------|-------|
| 1 | **Naive Fine-Tune** | Baseline (lower bound) | — |
| 2 | **Joint Training** | Baseline (upper bound / oracle) | — |
| 3 | **EWC** | Regularization | Kirkpatrick et al., PNAS 2017 |
| 4 | **Online EWC (EWC++)** | Regularization | Schwarz et al., ICML 2018 |
| 5 | **SI** | Regularization | Zenke et al., ICML 2017 |
| 6 | **LwF** | Distillation | Li & Hoiem, TPAMI 2017 |
| 7 | **DER++** | Replay | Buzzega et al., NeurIPS 2020 |

**Protocol:**  
- Each HARMONIC participant = 1 task (sequential sessions)  
- Policy: MLP mapping observation → joystick action (continuous regression)  
- Observation: `joint_positions` + `robot_position` (state features)  
- Action: `ada_joy` (joystick commands)  
- After each task, evaluate on ALL tasks → accuracy matrix R[i,j]  
- Metrics: Average MSE, Backward Transfer (BWT), Forward Transfer (FWT)  

All code is **inline** — no local package installation needed.

---

## 0. Environment Setup

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/thesis_project'
DATA_PROC    = f'{PROJECT_ROOT}/data/processed'
HARMONIC_DIR = f'{DATA_PROC}/harmonic'
RESULTS_DIR  = f'{PROJECT_ROOT}/experiments/exp01_il_comparison'

os.makedirs(RESULTS_DIR, exist_ok=True)

print(f'Processed data : {HARMONIC_DIR}')
print(f'Results dir    : {RESULTS_DIR}')

# Verify data exists
participants = sorted([d for d in os.listdir(HARMONIC_DIR) 
                       if os.path.isdir(os.path.join(HARMONIC_DIR, d)) and d.startswith('p')])
print(f'Participants   : {len(participants)} → {participants}')

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
!pip install -q pyarrow

import abc
import copy
import json
import logging
import random
import time
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import ConcatDataset, DataLoader, TensorDataset

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 120})

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(message)s')
logger = logging.getLogger(__name__)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__} — device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name()}')

PyTorch 2.10.0+cu128 — device: cuda
  GPU: Tesla T4


---
## 1. Metrics & Base Classes

All CL strategies share the same abstract interface: `on_task_start → train → on_task_end`.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Metrics
# ═══════════════════════════════════════════════════════════════

@dataclass
class TaskResult:
    """Evaluation results for one task."""
    task_id: int
    loss: float
    mse: float
    mae: float
    r2: float
    n_samples: int = 0


@dataclass
class ContinualMetrics:
    """
    Tracks the R[i,j] accuracy matrix.
    R[i][j] = MSE on task j after training up to task i.

    Derives:
      - Average Accuracy (AA): mean MSE on all tasks after final training
      - Backward Transfer (BWT): how much old tasks degrade (>0 = forgetting)
      - Forward Transfer (FWT): benefit of old knowledge on new tasks
    """
    n_tasks: int = 0
    accuracy_matrix: list = field(default_factory=list)
    task_names: list = field(default_factory=list)

    def record(self, trained_up_to, task_results):
        row = [r.mse for r in task_results]
        while len(self.accuracy_matrix) <= trained_up_to:
            self.accuracy_matrix.append([])
        self.accuracy_matrix[trained_up_to] = row

    @property
    def average_accuracy(self):
        if not self.accuracy_matrix: return float('nan')
        last = self.accuracy_matrix[-1]
        return float(np.mean(last)) if last else float('nan')

    @property
    def backward_transfer(self):
        T = len(self.accuracy_matrix)
        if T < 2: return 0.0
        bwt, cnt = 0.0, 0
        for j in range(T - 1):
            if j < len(self.accuracy_matrix[T-1]) and j < len(self.accuracy_matrix[j]):
                bwt += self.accuracy_matrix[T-1][j] - self.accuracy_matrix[j][j]
                cnt += 1
        return bwt / cnt if cnt > 0 else 0.0

    @property
    def forward_transfer(self):
        T = len(self.accuracy_matrix)
        if T < 2: return 0.0
        fwt, cnt = 0.0, 0
        for j in range(1, T):
            if j < len(self.accuracy_matrix[j-1]) and j < len(self.accuracy_matrix[j]):
                fwt += self.accuracy_matrix[j-1][j] - self.accuracy_matrix[j][j]
                cnt += 1
        return fwt / cnt if cnt > 0 else 0.0

    def summary_dict(self):
        return {
            'n_tasks': len(self.accuracy_matrix),
            'average_mse': self.average_accuracy,
            'backward_transfer': self.backward_transfer,
            'forward_transfer': self.forward_transfer,
            'accuracy_matrix': self.accuracy_matrix,
            'task_names': self.task_names,
        }

print('Metrics defined')

Metrics defined


In [ ]:
# ═══════════════════════════════════════════════════════════════
# MLP Policy Network
# ═══════════════════════════════════════════════════════════════

class MLPPolicy(nn.Module):
    """MLP: observation → action (continuous)."""

    def __init__(self, obs_dim, act_dim, hidden=(256, 256), dropout=0.1):
        super().__init__()
        self.obs_dim = obs_dim
        self.act_dim = act_dim
        layers = []
        d = obs_dim
        for h in hidden:
            layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(dropout)]
            d = h
        layers.append(nn.Linear(d, act_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

print('MLPPolicy defined')

MLPPolicy defined


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Abstract Continual Learner
# ═══════════════════════════════════════════════════════════════

class ContinualLearner(abc.ABC):
    """
    Base class for all CL strategies.  Subclasses implement:
      - on_task_start(task_id, loader)
      - compute_loss(obs, act, task_id) → scalar
      - on_task_end(task_id, loader)
    """
    def __init__(self, model, lr=1e-3, weight_decay=0., device='cpu'):
        self.model = model.to(device)
        self.device = device
        self.lr = lr
        self.weight_decay = weight_decay
        self.optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        self._current_task = -1

    @abc.abstractmethod
    def on_task_start(self, task_id, train_loader): ...

    @abc.abstractmethod
    def compute_loss(self, obs, act, task_id): ...

    @abc.abstractmethod
    def on_task_end(self, task_id, train_loader): ...

    def train_task(self, task_id, train_loader, val_loader=None,
                   epochs=50, patience=10, verbose=True):
        self._current_task = task_id
        self.on_task_start(task_id, train_loader)
        self.model.train()

        best_val, best_state, no_improve = float('inf'), None, 0
        history = {'train_loss': [], 'val_loss': []}

        for epoch in range(epochs):
            eloss, nb = 0., 0
            for batch in train_loader:
                obs, act = batch[0].to(self.device), batch[1].to(self.device)
                self.optimizer.zero_grad()
                loss = self.compute_loss(obs, act, task_id)
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                self.optimizer.step()
                eloss += loss.item(); nb += 1

            avg = eloss / max(nb, 1)
            history['train_loss'].append(avg)

            if val_loader is not None:
                vl = self._eval_loss(val_loader)
                history['val_loss'].append(vl)
                if vl < best_val:
                    best_val = vl
                    best_state = copy.deepcopy(self.model.state_dict())
                    no_improve = 0
                else:
                    no_improve += 1
                if verbose and (epoch+1) % 10 == 0:
                    print(f'    Ep {epoch+1:3d}/{epochs} | train {avg:.6f} | val {vl:.6f}')
                if no_improve >= patience:
                    if verbose: print(f'    Early stop at epoch {epoch+1}')
                    break
            else:
                if verbose and (epoch+1) % 10 == 0:
                    print(f'    Ep {epoch+1:3d}/{epochs} | train {avg:.6f}')

        if best_state is not None:
            self.model.load_state_dict(best_state)
        self.on_task_end(task_id, train_loader)
        return history

    def _eval_loss(self, loader):
        self.model.eval()
        tot, n = 0., 0
        with torch.no_grad():
            for b in loader:
                o, a = b[0].to(self.device), b[1].to(self.device)
                tot += F.mse_loss(self.model(o), a).item() * o.size(0)
                n += o.size(0)
        self.model.train()
        return tot / max(n, 1)

    def evaluate_task(self, test_loader, task_id):
        self.model.eval()
        preds, tgts = [], []
        with torch.no_grad():
            for b in test_loader:
                o, a = b[0].to(self.device), b[1].to(self.device)
                preds.append(self.model(o).cpu())
                tgts.append(a.cpu())
        p, t = torch.cat(preds), torch.cat(tgts)
        mse = F.mse_loss(p, t).item()
        mae = (p - t).abs().mean().item()
        ss_res = ((t - p)**2).sum().item()
        ss_tot = ((t - t.mean(0))**2).sum().item()
        r2 = 1 - ss_res / max(ss_tot, 1e-8)
        self.model.train()
        return TaskResult(task_id, mse, mse, mae, r2, len(t))

print('ContinualLearner base defined')

ContinualLearner base defined


---
## 2. CL Strategies (all 7 inline)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 1. Naive Fine-Tune (lower bound)
# ═══════════════════════════════════════════════════════════════

class NaiveFineTune(ContinualLearner):
    """No CL strategy — trains on each new task only. Catastrophic forgetting expected."""
    def on_task_start(self, task_id, train_loader):
        pass
    def compute_loss(self, obs, act, task_id):
        return F.mse_loss(self.model(obs), act)
    def on_task_end(self, task_id, train_loader):
        pass

# ═══════════════════════════════════════════════════════════════
# 2. Joint Training (upper bound / oracle)
# ═══════════════════════════════════════════════════════════════

class JointTraining(ContinualLearner):
    """Accumulates all data and retrains from scratch. Upper bound."""
    def __init__(self, model, lr=1e-3, device='cpu'):
        super().__init__(model, lr=lr, device=device)
        self._datasets = {}
        self._init_state = copy.deepcopy(model.state_dict())

    def on_task_start(self, task_id, train_loader):
        all_o, all_a = [], []
        for b in train_loader:
            all_o.append(b[0]); all_a.append(b[1])
        self._datasets[task_id] = TensorDataset(torch.cat(all_o), torch.cat(all_a))
        self.model.load_state_dict(copy.deepcopy(self._init_state))
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)

    def compute_loss(self, obs, act, task_id):
        return F.mse_loss(self.model(obs), act)

    def on_task_end(self, task_id, train_loader):
        pass

    def train_task(self, task_id, train_loader, val_loader=None,
                   epochs=50, patience=10, verbose=True):
        self._current_task = task_id
        self.on_task_start(task_id, train_loader)
        joint_ds = ConcatDataset(list(self._datasets.values()))
        bs = train_loader.batch_size if hasattr(train_loader, 'batch_size') else 256
        joint_loader = DataLoader(joint_ds, batch_size=bs, shuffle=True)
        self.model.train()
        return super().train_task(task_id, joint_loader, val_loader, epochs, patience, verbose)

print('Baselines (Naive + Joint) defined')

Baselines (Naive + Joint) defined


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 3+4. EWC / Online EWC  (Kirkpatrick 2017 / Schwarz 2018)
# ═══════════════════════════════════════════════════════════════

class EWC(ContinualLearner):
    """
    Elastic Weight Consolidation.
    Loss = L_task + (λ/2) Σ F_i (θ_i − θ*_i)²
    online=True → EWC++ (running Fisher average)
    """
    def __init__(self, model, lr=1e-3, device='cpu',
                 ewc_lambda=5000., fisher_samples=1000,
                 online=False, gamma=0.95):
        super().__init__(model, lr=lr, device=device)
        self.ewc_lambda = ewc_lambda
        self.fisher_samples = fisher_samples
        self.online = online
        self.gamma = gamma
        self._fishers = []
        self._opt_params = []
        self._run_fisher = None
        self._run_params = None

    def on_task_start(self, task_id, train_loader):
        pass

    def compute_loss(self, obs, act, task_id):
        pred = self.model(obs)
        loss = F.mse_loss(pred, act)
        if task_id == 0: return loss
        return loss + self._penalty()

    def on_task_end(self, task_id, train_loader):
        fisher = self._compute_fisher(train_loader)
        if self.online:
            if self._run_fisher is None:
                self._run_fisher = fisher
            else:
                for n in self._run_fisher:
                    self._run_fisher[n] = self.gamma * self._run_fisher[n] + fisher[n]
            self._run_params = {n: p.clone().detach() for n, p in self.model.named_parameters() if p.requires_grad}
        else:
            self._fishers.append(fisher)
            self._opt_params.append({n: p.clone().detach() for n, p in self.model.named_parameters() if p.requires_grad})

    def _compute_fisher(self, loader):
        fisher = {n: torch.zeros_like(p) for n, p in self.model.named_parameters() if p.requires_grad}
        self.model.eval()
        ns = 0
        for b in loader:
            o, a = b[0].to(self.device), b[1].to(self.device)
            self.model.zero_grad()
            F.mse_loss(self.model(o), a).backward()
            for n, p in self.model.named_parameters():
                if p.requires_grad and p.grad is not None:
                    fisher[n] += p.grad.data.pow(2) * o.size(0)
            ns += o.size(0)
            if ns >= self.fisher_samples: break
        for n in fisher: fisher[n] /= max(ns, 1)
        self.model.train()
        return fisher

    def _penalty(self):
        pen = torch.tensor(0., device=self.device)
        if self.online and self._run_fisher is not None:
            for n, p in self.model.named_parameters():
                if n in self._run_fisher:
                    pen += (self._run_fisher[n] * (p - self._run_params[n]).pow(2)).sum()
        else:
            for fi, op in zip(self._fishers, self._opt_params):
                for n, p in self.model.named_parameters():
                    if n in fi:
                        pen += (fi[n] * (p - op[n]).pow(2)).sum()
        return (self.ewc_lambda / 2.) * pen

print('EWC / EWC++ defined')

EWC / EWC++ defined


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 5. Synaptic Intelligence  (Zenke 2017)
# ═══════════════════════════════════════════════════════════════

class SI(ContinualLearner):
    """
    Synaptic Intelligence.
    Tracks online importance via path integral of gradient contributions.
    Loss = L_task + (c/2) Σ Ω_i (θ_i − θ*_i)²
    """
    def __init__(self, model, lr=1e-3, device='cpu', si_c=1.0, xi=1e-3):
        super().__init__(model, lr=lr, device=device)
        self.si_c = si_c
        self.xi = xi
        self._omega = {n: torch.zeros_like(p) for n, p in model.named_parameters() if p.requires_grad}
        self._prev   = {n: p.clone().detach()  for n, p in model.named_parameters() if p.requires_grad}
        self._w      = {n: torch.zeros_like(p) for n, p in model.named_parameters() if p.requires_grad}
        self._start  = {}

    def on_task_start(self, task_id, train_loader):
        for n, p in self.model.named_parameters():
            if p.requires_grad:
                self._start[n] = p.clone().detach()
                self._w[n] = torch.zeros_like(p)

    def compute_loss(self, obs, act, task_id):
        loss = F.mse_loss(self.model(obs), act)
        if task_id > 0:
            pen = torch.tensor(0., device=self.device)
            for n, p in self.model.named_parameters():
                if n in self._omega:
                    pen += (self._omega[n] * (p - self._prev[n]).pow(2)).sum()
            loss = loss + (self.si_c / 2.) * pen
        return loss

    def on_task_end(self, task_id, train_loader):
        for n, p in self.model.named_parameters():
            if p.requires_grad:
                delta = p.detach() - self._start[n]
                self._omega[n] += torch.clamp(self._w[n] / (delta.pow(2) + self.xi), min=0)
                self._prev[n] = p.clone().detach()

    def train_task(self, task_id, train_loader, val_loader=None,
                   epochs=50, patience=10, verbose=True):
        """Custom loop to track per-step gradient × param change."""
        self._current_task = task_id
        self.on_task_start(task_id, train_loader)
        self.model.train()

        best_val, best_state, no_improve = float('inf'), None, 0
        history = {'train_loss': [], 'val_loss': []}

        for epoch in range(epochs):
            eloss, nb = 0., 0
            for batch in train_loader:
                obs, act = batch[0].to(self.device), batch[1].to(self.device)
                prev_p = {n: p.clone().detach() for n, p in self.model.named_parameters() if p.requires_grad}

                self.optimizer.zero_grad()
                loss = self.compute_loss(obs, act, task_id)
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)

                for n, p in self.model.named_parameters():
                    if p.requires_grad and p.grad is not None:
                        self._w[n] += (-p.grad.detach()) * (p.detach() - prev_p[n] + 1e-20)

                self.optimizer.step()
                eloss += loss.item(); nb += 1

            avg = eloss / max(nb, 1)
            history['train_loss'].append(avg)

            if val_loader is not None:
                vl = self._eval_loss(val_loader)
                history['val_loss'].append(vl)
                if vl < best_val:
                    best_val = vl; best_state = copy.deepcopy(self.model.state_dict()); no_improve = 0
                else:
                    no_improve += 1
                if verbose and (epoch+1) % 10 == 0:
                    print(f'    Ep {epoch+1:3d}/{epochs} | train {avg:.6f} | val {vl:.6f}')
                if no_improve >= patience:
                    if verbose: print(f'    Early stop at epoch {epoch+1}')
                    break
            else:
                if verbose and (epoch+1) % 10 == 0:
                    print(f'    Ep {epoch+1:3d}/{epochs} | train {avg:.6f}')

        if best_state is not None:
            self.model.load_state_dict(best_state)
        self.on_task_end(task_id, train_loader)
        return history

print('SI defined')

SI defined


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 6. Learning without Forgetting  (Li & Hoiem 2017)
# ═══════════════════════════════════════════════════════════════

class LwF(ContinualLearner):
    """
    Knowledge distillation from a teacher (model snapshot before new task).
    Loss = L_task + α * MSE(student, teacher)
    """
    def __init__(self, model, lr=1e-3, device='cpu', lwf_alpha=1.0):
        super().__init__(model, lr=lr, device=device)
        self.lwf_alpha = lwf_alpha
        self._teacher = None

    def on_task_start(self, task_id, train_loader):
        if task_id > 0:
            self._teacher = copy.deepcopy(self.model)
            self._teacher.eval()
            for p in self._teacher.parameters(): p.requires_grad = False
        else:
            self._teacher = None

    def compute_loss(self, obs, act, task_id):
        pred = self.model(obs)
        loss = F.mse_loss(pred, act)
        if self._teacher is not None and task_id > 0:
            with torch.no_grad():
                teacher_pred = self._teacher(obs)
            loss = loss + self.lwf_alpha * F.mse_loss(pred, teacher_pred)
        return loss

    def on_task_end(self, task_id, train_loader):
        pass

print('LwF defined')

LwF defined


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 7. DER++  (Buzzega et al. NeurIPS 2020)
# ═══════════════════════════════════════════════════════════════

class ReplayBuffer:
    """Fixed-size reservoir-sampling replay buffer."""
    def __init__(self, capacity=5000):
        self.capacity = capacity
        self.obs, self.act, self.logits = [], [], []
        self._n = 0

    def __len__(self): return len(self.obs)

    def add(self, obs, act, logits):
        for i in range(obs.size(0)):
            self._n += 1
            if len(self.obs) < self.capacity:
                self.obs.append(obs[i].cpu())
                self.act.append(act[i].cpu())
                self.logits.append(logits[i].cpu())
            else:
                j = random.randint(0, self._n - 1)
                if j < self.capacity:
                    self.obs[j] = obs[i].cpu()
                    self.act[j] = act[i].cpu()
                    self.logits[j] = logits[i].cpu()

    def sample(self, n):
        n = min(n, len(self.obs))
        idx = random.sample(range(len(self.obs)), n)
        return (torch.stack([self.obs[i] for i in idx]),
                torch.stack([self.act[i] for i in idx]),
                torch.stack([self.logits[i] for i in idx]))


class DERPlusPlus(ContinualLearner):
    """
    Dark Experience Replay++.
    Loss = L_task + α ∙ MSE(pred, stored_logits) + β ∙ MSE(pred, stored_labels)
    """
    def __init__(self, model, lr=1e-3, device='cpu',
                 buffer_size=5000, alpha=0.5, beta=0.5):
        super().__init__(model, lr=lr, device=device)
        self.buffer = ReplayBuffer(buffer_size)
        self.alpha = alpha
        self.beta = beta

    def on_task_start(self, task_id, train_loader):
        pass

    def compute_loss(self, obs, act, task_id):
        pred = self.model(obs)
        loss = F.mse_loss(pred, act)
        with torch.no_grad():
            logits = self.model(obs).detach()
        self.buffer.add(obs.detach(), act.detach(), logits)

        if len(self.buffer) > 0 and task_id > 0:
            n = max(1, obs.size(0))
            bo, ba, bl = self.buffer.sample(n)
            bo, ba, bl = bo.to(self.device), ba.to(self.device), bl.to(self.device)
            bp = self.model(bo)
            loss = loss + self.alpha * F.mse_loss(bp, bl) + self.beta * F.mse_loss(bp, ba)
        return loss

    def on_task_end(self, task_id, train_loader):
        pass

print('DER++ defined')

DER++ defined


---
## 3. Data Loading

Load preprocessed HARMONIC Parquet files from Google Drive.  
Each participant = 1 task; observation = `joint_positions` + `robot_position`, action = `ada_joy`.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Data structures
# ═══════════════════════════════════════════════════════════════

@dataclass
class TaskData:
    task_id: int
    participant_id: str
    obs_train: torch.Tensor
    act_train: torch.Tensor
    obs_val: torch.Tensor
    act_val: torch.Tensor
    obs_test: torch.Tensor
    act_test: torch.Tensor

    @property
    def n_train(self): return self.obs_train.shape[0]
    @property
    def n_val(self): return self.obs_val.shape[0]
    @property
    def n_test(self): return self.obs_test.shape[0]

    def train_loader(self, bs=256):
        return DataLoader(TensorDataset(self.obs_train, self.act_train), batch_size=bs, shuffle=True)
    def val_loader(self, bs=512):
        return DataLoader(TensorDataset(self.obs_val, self.act_val), batch_size=bs)
    def test_loader(self, bs=512):
        return DataLoader(TensorDataset(self.obs_test, self.act_test), batch_size=bs)

print('TaskData defined')

TaskData defined


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Load one participant's data from Parquet
# ═══════════════════════════════════════════════════════════════

def load_participant(processed_dir, participant,
                     state_mods=('joint_positions', 'robot_position'),
                     action_mod='ada_joy',
                     val_ratio=0.15, seed=42):
    """
    Load all preprocessed trial Parquet files for one participant.
    Returns (obs_train, act_train, obs_val, act_val, obs_test, act_test) as tensors.
    """
    pdir = Path(processed_dir) / participant
    assert pdir.exists(), f'Not found: {pdir}'

    all_obs, all_act = [], []
    for trial_dir in sorted(pdir.iterdir()):
        if not trial_dir.is_dir(): continue
        act_path = trial_dir / f'{action_mod}.parquet'
        if not act_path.exists(): continue

        act_df = pd.read_parquet(act_path)
        act_cols = [c for c in act_df.columns if c not in ('time_s', 'timestamp', 'rosbag_timestamp')]
        if not act_cols: continue
        act_arr = act_df[act_cols].values

        state_parts = []
        skip = False
        for mod in state_mods:
            mp = trial_dir / f'{mod}.parquet'
            if not mp.exists(): skip = True; break
            df = pd.read_parquet(mp)
            cols = [c for c in df.columns if c not in ('time_s', 'timestamp', 'rosbag_timestamp')]
            state_parts.append(df[cols].values)
        if skip or not state_parts: continue

        obs_arr = np.concatenate(state_parts, axis=1)
        n = min(obs_arr.shape[0], act_arr.shape[0])
        obs_arr, act_arr = obs_arr[:n], act_arr[:n]
        valid = ~(np.isnan(obs_arr).any(1) | np.isnan(act_arr).any(1))
        obs_arr, act_arr = obs_arr[valid], act_arr[valid]
        if len(obs_arr) > 0:
            all_obs.append(obs_arr)
            all_act.append(act_arr)

    assert all_obs, f'No data for {participant}'
    obs = np.nan_to_num(np.concatenate(all_obs).astype(np.float32))
    act = np.nan_to_num(np.concatenate(all_act).astype(np.float32))

    rng = np.random.RandomState(seed)
    idx = rng.permutation(len(obs))
    nv = int(len(obs) * val_ratio)
    nt = nv
    obs_t, act_t = torch.from_numpy(obs), torch.from_numpy(act)
    return (obs_t[idx[nt+nv:]], act_t[idx[nt+nv:]],
            obs_t[idx[nt:nt+nv]], act_t[idx[nt:nt+nv]],
            obs_t[idx[:nt]], act_t[idx[:nt]])

print('load_participant defined')

load_participant defined


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Build the task sequence (one participant = one task)
# ═══════════════════════════════════════════════════════════════

# Configuration
STATE_MODS   = ('joint_positions', 'robot_position')  # obs features
ACTION_MOD   = 'ada_joy'                               # target
N_TASKS      = min(len(participants), 10)  # Use up to 10 for manageable runtime
SEED         = 42

task_pids = participants[:N_TASKS]
print(f'Building {N_TASKS} tasks from participants: {task_pids}')

tasks = []
for tid, pid in enumerate(task_pids):
    try:
        data = load_participant(HARMONIC_DIR, pid, STATE_MODS, ACTION_MOD, seed=SEED)
        td = TaskData(tid, pid, *data)
        tasks.append(td)
        print(f'  Task {tid:2d} ({pid}): train={td.n_train:5d}  val={td.n_val:4d}  test={td.n_test:4d}')
    except Exception as e:
        print(f'  SKIP {pid}: {e}')

obs_dim = tasks[0].obs_train.shape[1]
act_dim = tasks[0].act_train.shape[1]
print(f'\nobs_dim={obs_dim}, act_dim={act_dim}, n_tasks={len(tasks)}')

Building 10 tasks from participants: ['p100', 'p101', 'p102', 'p103', 'p104', 'p105', 'p106', 'p107', 'p108', 'p109']
  Task  0 (p100): train=21274  val=4558  test=4558
  Task  1 (p101): train=26797  val=5742  test=5742
  Task  2 (p102): train=33573  val=7194  test=7194
  Task  3 (p103): train=16945  val=3631  test=3631
  Task  4 (p104): train=17868  val=3828  test=3828
  SKIP p105: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 1, the array at index 0 has size 51 and the array at index 9 has size 55
  Task  6 (p106): train=31734  val=6800  test=6800
  Task  7 (p107): train=14412  val=3087  test=3087
  Task  8 (p108): train=10253  val=2196  test=2196
  Task  9 (p109): train=21081  val=4516  test=4516

obs_dim=51, act_dim=5, n_tasks=9


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Global z-score normalization (using train data only)
# ═══════════════════════════════════════════════════════════════

all_obs_train = torch.cat([t.obs_train for t in tasks])
all_act_train = torch.cat([t.act_train for t in tasks])

obs_mean = all_obs_train.mean(0)
obs_std  = all_obs_train.std(0).clamp(min=1e-6)
act_mean = all_act_train.mean(0)
act_std  = all_act_train.std(0).clamp(min=1e-6)

for t in tasks:
    t.obs_train = (t.obs_train - obs_mean) / obs_std
    t.obs_val   = (t.obs_val   - obs_mean) / obs_std
    t.obs_test  = (t.obs_test  - obs_mean) / obs_std
    t.act_train = (t.act_train - act_mean) / act_std
    t.act_val   = (t.act_val   - act_mean) / act_std
    t.act_test  = (t.act_test  - act_mean) / act_std

del all_obs_train, all_act_train
print(f'Normalized. obs_mean shape: {obs_mean.shape}, act_mean shape: {act_mean.shape}')

Normalized. obs_mean shape: torch.Size([51]), act_mean shape: torch.Size([5])


---
## 4. Benchmark Runner

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Run one strategy across the full task sequence
# ═══════════════════════════════════════════════════════════════

def run_benchmark(name, learner, tasks, epochs=50, patience=10, bs=256, verbose=True):
    metrics = ContinualMetrics(n_tasks=len(tasks),
                               task_names=[t.participant_id for t in tasks])
    histories = []
    t0 = time.time()

    for task in tasks:
        if verbose:
            print(f'\n{"="*55}')
            print(f'[{name}] Task {task.task_id} ({task.participant_id}) — '
                  f'train={task.n_train}, val={task.n_val}')
            print(f'{"="*55}')

        h = learner.train_task(
            task.task_id,
            task.train_loader(bs),
            task.val_loader(bs*2),
            epochs=epochs, patience=patience, verbose=verbose
        )
        histories.append(h)

        # Evaluate on ALL tasks
        results = []
        for et in tasks:
            tr = learner.evaluate_task(et.test_loader(bs*2), et.task_id)
            results.append(tr)
            if verbose and et.task_id <= task.task_id:
                tag = 'CUR' if et.task_id == task.task_id else 'OLD'
                print(f'  [{tag}] Task {et.task_id} ({et.participant_id}): '
                      f'MSE={tr.mse:.6f}  MAE={tr.mae:.4f}  R²={tr.r2:.4f}')
        metrics.record(task.task_id, results)

    elapsed = time.time() - t0
    if verbose:
        print(f'\n{"="*55}')
        print(f'[{name}] Done in {elapsed:.1f}s')
        print(f'  Avg MSE : {metrics.average_accuracy:.6f}')
        print(f'  BWT     : {metrics.backward_transfer:+.6f}')
        print(f'  FWT     : {metrics.forward_transfer:+.6f}')
        print(f'{"="*55}')

    return {'name': name, 'metrics': metrics, 'time': elapsed, 'histories': histories}

print('Benchmark runner defined')

Benchmark runner defined


---
## 5. Run All Strategies

**Hyperparameters** (kept consistent for fair comparison):

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Hyperparameters
# ═══════════════════════════════════════════════════════════════

HIDDEN      = (256, 256)
LR          = 1e-3
DROPOUT     = 0.1
EPOCHS      = 50
PATIENCE    = 10
BATCH_SIZE  = 256

# Strategy-specific
EWC_LAMBDA      = 5000.0
EWC_FISHER_N    = 1000
EWCPP_GAMMA     = 0.95
SI_C            = 1.0
LWF_ALPHA       = 1.0
DER_BUFFER      = 5000
DER_ALPHA       = 0.5
DER_BETA        = 0.5

config = {
    'hidden': HIDDEN, 'lr': LR, 'dropout': DROPOUT,
    'epochs': EPOCHS, 'patience': PATIENCE, 'batch_size': BATCH_SIZE,
    'ewc_lambda': EWC_LAMBDA, 'ewc_fisher_n': EWC_FISHER_N,
    'ewcpp_gamma': EWCPP_GAMMA, 'si_c': SI_C, 'lwf_alpha': LWF_ALPHA,
    'der_buffer': DER_BUFFER, 'der_alpha': DER_ALPHA, 'der_beta': DER_BETA,
    'obs_dim': obs_dim, 'act_dim': act_dim, 'n_tasks': len(tasks),
    'state_modalities': list(STATE_MODS), 'action_modality': ACTION_MOD,
    'seed': SEED, 'device': DEVICE,
}

# Save config
with open(f'{RESULTS_DIR}/config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('Config saved. Starting benchmark...')
print(json.dumps(config, indent=2))

Config saved. Starting benchmark...
{
  "hidden": [
    256,
    256
  ],
  "lr": 0.001,
  "dropout": 0.1,
  "epochs": 50,
  "patience": 10,
  "batch_size": 256,
  "ewc_lambda": 5000.0,
  "ewc_fisher_n": 1000,
  "ewcpp_gamma": 0.95,
  "si_c": 1.0,
  "lwf_alpha": 1.0,
  "der_buffer": 5000,
  "der_alpha": 0.5,
  "der_beta": 0.5,
  "obs_dim": 51,
  "act_dim": 5,
  "n_tasks": 9,
  "state_modalities": [
    "joint_positions",
    "robot_position"
  ],
  "action_modality": "ada_joy",
  "seed": 42,
  "device": "cuda"
}


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Helper: fresh model factory + results storage
# ═══════════════════════════════════════════════════════════════

def make_model():
    return MLPPolicy(obs_dim, act_dim, HIDDEN, DROPOUT)

def run_and_save(name, learner):
    """Run benchmark for one strategy, save results, store in all_results."""
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    random.seed(SEED)

    result = run_benchmark(name, learner, tasks,
                           epochs=EPOCHS, patience=PATIENCE,
                           bs=BATCH_SIZE, verbose=True)
    all_results[name] = result

    # Save per-strategy results to Drive
    fname = name.lower().replace(' ', '_').replace('+', 'p')
    with open(f'{RESULTS_DIR}/{fname}_results.json', 'w') as f:
        json.dump(result['metrics'].summary_dict(), f, indent=2, default=str)
    return result

# Accumulated results across all strategy cells
all_results = {}

print(f'Ready — obs_dim={obs_dim}, act_dim={act_dim}, n_tasks={len(tasks)}')

Strategies to benchmark: ['Naive Fine-Tune', 'Joint Training', 'EWC', 'Online EWC', 'SI', 'LwF', 'DER++']


### 5.1 — Naive Fine-Tune (Lower Bound)
No continual learning mechanism. The model is simply trained on each new task, overwriting what it learned before.  
**Expected:** Severe catastrophic forgetting → high BWT.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 1. Naive Fine-Tune (lower bound — catastrophic forgetting baseline)
# ═══════════════════════════════════════════════════════════════

learner = NaiveFineTune(make_model(), lr=LR, device=DEVICE)
res_naive = run_and_save('Naive Fine-Tune', learner)

print(f"\n{'─'*40}")
print(f"Avg MSE : {res_naive['metrics'].average_accuracy:.6f}")
print(f"BWT     : {res_naive['metrics'].backward_transfer:+.6f}")
print(f"FWT     : {res_naive['metrics'].forward_transfer:+.6f}")
print(f"Time    : {res_naive['time']:.1f}s")

### 5.2 — Joint Training (Upper Bound / Oracle)
Accumulates ALL prior task data and retrains the model from scratch on the union.  
**Expected:** Best performance (no forgetting by construction), but computationally expensive and not truly incremental.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 2. Joint Training (upper bound — oracle retrain on all data)
# ═══════════════════════════════════════════════════════════════

learner = JointTraining(make_model(), lr=LR, device=DEVICE)
res_joint = run_and_save('Joint Training', learner)

print(f"\n{'─'*40}")
print(f"Avg MSE : {res_joint['metrics'].average_accuracy:.6f}")
print(f"BWT     : {res_joint['metrics'].backward_transfer:+.6f}")
print(f"FWT     : {res_joint['metrics'].forward_transfer:+.6f}")
print(f"Time    : {res_joint['time']:.1f}s")

### 5.3 — EWC (Elastic Weight Consolidation)
Kirkpatrick et al., PNAS 2017.  
Adds a Fisher Information penalty: $\mathcal{L} = \mathcal{L}_{task} + \frac{\lambda}{2} \sum_i F_i (\theta_i - \theta_i^*)^2$  
**Expected:** Reduced forgetting vs Naive, but Fisher is computed per-task (memory grows with T).

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 3. EWC — Elastic Weight Consolidation (Kirkpatrick 2017)
# ═══════════════════════════════════════════════════════════════

learner = EWC(make_model(), lr=LR, device=DEVICE,
              ewc_lambda=EWC_LAMBDA, fisher_samples=EWC_FISHER_N)
res_ewc = run_and_save('EWC', learner)

print(f"\n{'─'*40}")
print(f"Avg MSE : {res_ewc['metrics'].average_accuracy:.6f}")
print(f"BWT     : {res_ewc['metrics'].backward_transfer:+.6f}")
print(f"FWT     : {res_ewc['metrics'].forward_transfer:+.6f}")
print(f"Time    : {res_ewc['time']:.1f}s")

### 5.4 — Online EWC (EWC++)
Schwarz et al., ICML 2018.  
Uses a **running average** of the Fisher matrix instead of storing one per task: $\hat{F} = \gamma \hat{F}_{prev} + F_{new}$  
**Expected:** Similar to EWC but with constant memory footprint.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 4. Online EWC / EWC++ (Schwarz 2018)
# ═══════════════════════════════════════════════════════════════

learner = EWC(make_model(), lr=LR, device=DEVICE,
              ewc_lambda=EWC_LAMBDA, fisher_samples=EWC_FISHER_N,
              online=True, gamma=EWCPP_GAMMA)
res_oewc = run_and_save('Online EWC', learner)

print(f"\n{'─'*40}")
print(f"Avg MSE : {res_oewc['metrics'].average_accuracy:.6f}")
print(f"BWT     : {res_oewc['metrics'].backward_transfer:+.6f}")
print(f"FWT     : {res_oewc['metrics'].forward_transfer:+.6f}")
print(f"Time    : {res_oewc['time']:.1f}s")

### 5.5 — Synaptic Intelligence (SI)
Zenke et al., ICML 2017.  
Tracks per-parameter importance $\Omega$ online via path integral of gradient contributions during training.  
$\mathcal{L} = \mathcal{L}_{task} + \frac{c}{2} \sum_i \Omega_i (\theta_i - \theta_i^*)^2$  
**Expected:** Comparable to EWC but with online importance estimation (no separate Fisher pass).

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 5. Synaptic Intelligence (Zenke 2017)
# ═══════════════════════════════════════════════════════════════

learner = SI(make_model(), lr=LR, device=DEVICE, si_c=SI_C)
res_si = run_and_save('SI', learner)

print(f"\n{'─'*40}")
print(f"Avg MSE : {res_si['metrics'].average_accuracy:.6f}")
print(f"BWT     : {res_si['metrics'].backward_transfer:+.6f}")
print(f"FWT     : {res_si['metrics'].forward_transfer:+.6f}")
print(f"Time    : {res_si['time']:.1f}s")

### 5.6 — Learning without Forgetting (LwF)
Li & Hoiem, TPAMI 2017.  
At each new task, snapshots the model as a frozen **teacher** and adds a distillation loss:  
$\mathcal{L} = \mathcal{L}_{task} + \alpha \cdot \text{MSE}(\hat{y}_{student}, \hat{y}_{teacher})$  
**Expected:** Good when task domains are similar; may struggle with very different participants.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 6. LwF — Learning without Forgetting (Li & Hoiem 2017)
# ═══════════════════════════════════════════════════════════════

learner = LwF(make_model(), lr=LR, device=DEVICE, lwf_alpha=LWF_ALPHA)
res_lwf = run_and_save('LwF', learner)

print(f"\n{'─'*40}")
print(f"Avg MSE : {res_lwf['metrics'].average_accuracy:.6f}")
print(f"BWT     : {res_lwf['metrics'].backward_transfer:+.6f}")
print(f"FWT     : {res_lwf['metrics'].forward_transfer:+.6f}")
print(f"Time    : {res_lwf['time']:.1f}s")

### 5.7 — DER++ (Dark Experience Replay++)
Buzzega et al., NeurIPS 2020.  
Uses a **replay buffer** (reservoir sampling) storing past observations + their logits. Adds two replay losses:  
$\mathcal{L} = \mathcal{L}_{task} + \alpha \cdot \text{MSE}(\hat{y}, \text{stored\_logits}) + \beta \cdot \text{MSE}(\hat{y}, \text{stored\_labels})$  
**Expected:** Strong anti-forgetting from replay; typically best among compared methods.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 7. DER++ — Dark Experience Replay++ (Buzzega 2020)
# ═══════════════════════════════════════════════════════════════

learner = DERPlusPlus(make_model(), lr=LR, device=DEVICE,
                      buffer_size=DER_BUFFER,
                      alpha=DER_ALPHA, beta=DER_BETA)
res_der = run_and_save('DER++', learner)

print(f"\n{'─'*40}")
print(f"Avg MSE : {res_der['metrics'].average_accuracy:.6f}")
print(f"BWT     : {res_der['metrics'].backward_transfer:+.6f}")
print(f"FWT     : {res_der['metrics'].forward_transfer:+.6f}")
print(f"Time    : {res_der['time']:.1f}s")

---
## 6. Results & Comparison

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Summary table
# ═══════════════════════════════════════════════════════════════

rows = []
for name, res in all_results.items():
    m = res['metrics']
    rows.append({
        'Strategy': name,
        'Avg MSE ↓': f'{m.average_accuracy:.6f}',
        'BWT →0': f'{m.backward_transfer:+.6f}',
        'FWT ↓': f'{m.forward_transfer:+.6f}',
        'Time (s)': f'{res["time"]:.1f}',
    })

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

# Save
summary_df.to_csv(f'{RESULTS_DIR}/comparison_table.csv', index=False)
display(summary_df)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Bar chart: Avg MSE comparison
# ═══════════════════════════════════════════════════════════════

names = list(all_results.keys())
mses  = [all_results[n]['metrics'].average_accuracy for n in names]
bwts  = [all_results[n]['metrics'].backward_transfer for n in names]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Avg MSE
colors = ['#e74c3c', '#2ecc71', '#3498db', '#9b59b6', '#f39c12', '#1abc9c', '#e67e22']
ax = axes[0]
bars = ax.barh(names, mses, color=colors[:len(names)])
ax.set_xlabel('Average MSE (lower = better)')
ax.set_title('Average MSE After All Tasks')
ax.invert_yaxis()
for bar, v in zip(bars, mses):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{v:.4f}', va='center', fontsize=9)

# BWT
ax = axes[1]
bars = ax.barh(names, bwts, color=colors[:len(names)])
ax.set_xlabel('Backward Transfer (closer to 0 = less forgetting)')
ax.set_title('Backward Transfer (BWT)')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.invert_yaxis()
for bar, v in zip(bars, bwts):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{v:+.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/comparison_bars.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Accuracy matrices (heatmaps)
# ═══════════════════════════════════════════════════════════════

n_strats = len(all_results)
fig, axes = plt.subplots(2, (n_strats + 1) // 2, figsize=(5 * ((n_strats+1)//2), 8))
axes = axes.flatten()

for idx, (name, res) in enumerate(all_results.items()):
    mat = np.array(res['metrics'].accuracy_matrix)
    ax = axes[idx]
    im = ax.imshow(mat, cmap='YlOrRd', aspect='auto')
    ax.set_title(name, fontsize=10)
    ax.set_xlabel('Eval task')
    ax.set_ylabel('Trained up to')
    ax.set_xticks(range(mat.shape[1]))
    ax.set_yticks(range(mat.shape[0]))
    plt.colorbar(im, ax=ax, shrink=0.6)

# Hide unused axes
for idx in range(n_strats, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('R[i,j] Accuracy Matrix — MSE on task j after training up to task i', fontsize=12)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/accuracy_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Performance on first task over time (forgetting curve)
# ═══════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(10, 5))

for idx, (name, res) in enumerate(all_results.items()):
    mat = res['metrics'].accuracy_matrix
    task0_perf = [mat[i][0] for i in range(len(mat))]
    ax.plot(range(len(task0_perf)), task0_perf, 'o-',
            label=name, color=colors[idx], linewidth=2, markersize=5)

ax.set_xlabel('After training on task i')
ax.set_ylabel('MSE on Task 0 (first participant)')
ax.set_title('Forgetting Curve — Performance on Task 0 Over Time')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/forgetting_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Average MSE evolution (on all seen tasks)
# ═══════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(10, 5))

for idx, (name, res) in enumerate(all_results.items()):
    mat = res['metrics'].accuracy_matrix
    avg_per_step = []
    for i in range(len(mat)):
        seen = mat[i][:i+1]  # only tasks seen so far
        avg_per_step.append(np.mean(seen))
    ax.plot(range(len(avg_per_step)), avg_per_step, 'o-',
            label=name, color=colors[idx], linewidth=2, markersize=5)

ax.set_xlabel('After training on task i')
ax.set_ylabel('Avg MSE on tasks 0..i')
ax.set_title('Average MSE on Seen Tasks Over Training Sequence')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/avg_mse_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Per-task final R² scores
# ═══════════════════════════════════════════════════════════════

# Re-evaluate all strategies on all tasks to get R² (stored in accuracy matrix as MSE)
# We'll compute R² from the last row of accuracy matrix for each strategy

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(tasks))
w = 0.8 / len(all_results)

for idx, (name, res) in enumerate(all_results.items()):
    mat = np.array(res['metrics'].accuracy_matrix)
    last_row = mat[-1]  # MSE on each task after all training
    ax.bar(x + idx * w, last_row, w, label=name, color=colors[idx], alpha=0.85)

ax.set_xlabel('Task (participant)')
ax.set_ylabel('MSE')
ax.set_title('Final MSE Per Task (after training on all tasks)')
ax.set_xticks(x + w * len(all_results) / 2)
ax.set_xticklabels([t.participant_id for t in tasks], rotation=45)
ax.legend(fontsize=8, loc='upper right')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/per_task_final_mse.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Save All Results to Drive

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Save comprehensive results
# ═══════════════════════════════════════════════════════════════

# Full comparison JSON
comparison = {}
for name, res in all_results.items():
    comparison[name] = {
        'avg_mse': res['metrics'].average_accuracy,
        'bwt': res['metrics'].backward_transfer,
        'fwt': res['metrics'].forward_transfer,
        'time_s': round(res['time'], 1),
        'accuracy_matrix': res['metrics'].accuracy_matrix,
    }

with open(f'{RESULTS_DIR}/full_comparison.json', 'w') as f:
    json.dump(comparison, f, indent=2)

# Save normalization stats for future use
norm_stats = {
    'obs_mean': obs_mean.tolist(),
    'obs_std': obs_std.tolist(),
    'act_mean': act_mean.tolist(),
    'act_std': act_std.tolist(),
}
with open(f'{RESULTS_DIR}/normalization_stats.json', 'w') as f:
    json.dump(norm_stats, f, indent=2)

print('Results saved to:', RESULTS_DIR)
print('\nFiles:')
for f in sorted(os.listdir(RESULTS_DIR)):
    size = os.path.getsize(os.path.join(RESULTS_DIR, f))
    print(f'  {f:40s} {size/1024:.1f} KB')

---
## 8. Summary & Interpretation

### Expected Outcomes

| Strategy | Expected BWT | Why |
|----------|-------------|-----|
| **Naive Fine-Tune** | Large positive (bad) | No protection against forgetting |
| **Joint Training** | ≈ 0 (best) | Retrains on all data (oracle) |
| **EWC** | Moderate | Fisher-based importance prevents large changes |
| **Online EWC** | Moderate (better than EWC) | Running Fisher avoids memory growth |
| **SI** | Moderate | Online importance via path integral |
| **LwF** | Moderate | Distillation preserves output distribution |
| **DER++** | Small (good) | Replay buffer + dark knowledge |

### Key Research Questions
1. Does DER++ outperform regularization methods (EWC, SI, LwF) on HRC action prediction?
2. How does inter-participant variability affect forgetting patterns?
3. Is Online EWC significantly better than classical EWC for this sequential setting?

### Next Steps
- **Hyperparameter sweep**: vary λ (EWC), c (SI), α (LwF), buffer size (DER++)  
- **Add PackNet**: parameter isolation strategy (adapter-based)  
- **Experiment 2**: Combine with IRL for online preference learning  
- **Experiment 3**: Incorporate style vectors from DASIG into conditioning  